# 05 — EDAT Training (Colab, Drive-connected)

Two tasks:
1. **Epsilon tuning** — grid-search `edat_epsilon` on val set to find best perturbation magnitude
2. **EDAT-integrated training** — full training loop with adversarial perturbations in embedding space

Builds on `04_ultimate_training.ipynb` architecture (LoRA + focal+LS loss + EMA).  
Designed for Colab T4/A100. Data loaded directly from Google Drive.

## 0 — Install

In [1]:
import subprocess, sys
pkgs = [
    "transformers>=4.40", "peft>=0.10", "torch",
    "pyyaml", "scikit-learn", "tqdm", "bitsandbytes",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
print("Done.")

Done.


## 1 — Mount Drive + Config

In [2]:
import os, json, random, sys, time
from pathlib import Path
import yaml
import torch
import numpy as np

# ── Mount Drive ──────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/CSI_Project")
    IS_COLAB = True
    print("Colab — Drive mounted")
except ImportError:
    BASE_DIR = Path.cwd()
    IS_COLAB = False
    print(f"Local — BASE_DIR: {BASE_DIR}")

# ── Load config ──────────────────────────────────────────────────────────────
with open(BASE_DIR / "config.yaml") as f:
    cfg = yaml.safe_load(f)

# ── Colab-optimised overrides ────────────────────────────────────────────────
MAX_LENGTH      = 128          # small patches — fits large batches on T4
BATCH_SIZE      = 32           # increase if GPU mem allows
EPOCHS          = 8
LR              = 2e-4
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.10
LABEL_SMOOTHING = 0.05
FOCAL_GAMMA     = 2.0
EMA_DECAY       = 0.999
MAX_GRAD_NORM   = 1.0
PATIENCE        = 4
MS_DROPOUT_N    = 5

# ── EDAT hyper-parameters (epsilon tuned in Section 3) ──────────────────────
EDAT_STEPS      = int(cfg.get("edat_steps", 3))       # PGD steps
EDAT_STEP_SIZE  = float(cfg.get("edat_step_size", 0.005))
EDAT_KL_WEIGHT  = float(cfg.get("edat_kl_weight", 1.0))
# epsilon set after tuning — placeholder here
EDAT_EPSILON    = float(cfg.get("edat_epsilon", 0.01))

# ── Paths ────────────────────────────────────────────────────────────────────
TOKEN_CACHE_DIR = BASE_DIR / cfg["token_cache_dir"]
CHECKPOINT_DIR  = BASE_DIR / cfg["checkpoint_dir"]
LOG_DIR         = BASE_DIR / cfg["log_dir"]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Cache was built with max_length=512 — we slice to MAX_LENGTH at load time
CACHE_FILE = TOKEN_CACHE_DIR / f"tokens_maxlen{cfg['max_length']}.pt"
assert CACHE_FILE.exists(), f"Missing cache: {CACHE_FILE}"

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

SEED = cfg["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == "cuda": torch.cuda.manual_seed_all(SEED)

print(f"Device     : {DEVICE}")
print(f"MAX_LENGTH : {MAX_LENGTH}")
print(f"BATCH_SIZE : {BATCH_SIZE}")
print(f"EDAT eps   : {EDAT_EPSILON} (will be tuned in §3)")

Mounted at /content/drive
Colab — Drive mounted
Device     : cuda
MAX_LENGTH : 128
BATCH_SIZE : 32
EDAT eps   : 0.01 (will be tuned in §3)


## 2 — Dataset + DataLoaders

In [3]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class VulnerabilityDataset(Dataset):
    def __init__(self, cache, split, max_length=128):
        idx = [i for i, s in enumerate(cache["split_origins"]) if s == split]
        self.input_ids      = cache["input_ids"][idx][:, :max_length].contiguous()
        self.attention_mask = cache["attention_mask"][idx][:, :max_length].contiguous()
        self.cwe_labels     = cache["cwe_labels"][idx]
        self.binary_labels  = cache["binary_labels"][idx]

    def __len__(self): return len(self.input_ids)

    def __getitem__(self, i):
        return {
            "input_ids":      self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            "cwe_label":      self.cwe_labels[i],
        }


def make_balanced_sampler(ds):
    labels = ds.cwe_labels.numpy()
    counts = np.bincount(labels, minlength=8)
    w  = 1.0 / (counts + 1e-6)
    sw = w[labels]
    return WeightedRandomSampler(torch.tensor(sw, dtype=torch.float),
                                 num_samples=len(labels), replacement=True)


cache = torch.load(CACHE_FILE, weights_only=True)
print(f"Cache: {cache['num_records']:,} records")

_kw = dict(num_workers=2, pin_memory=(DEVICE == "cuda"),
           persistent_workers=False, prefetch_factor=2)

train_ds = VulnerabilityDataset(cache, "train", MAX_LENGTH)
val_ds   = VulnerabilityDataset(cache, "val",   MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=make_balanced_sampler(train_ds), **_kw)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **_kw)

# Class weights: (1 - freq)^0.5, mean-normalised
_counts = np.bincount(train_ds.cwe_labels.numpy(), minlength=8).astype(float)
_freq   = _counts / _counts.sum()
CLASS_WEIGHTS = torch.tensor(np.sqrt(1.0 - _freq + 1e-6), dtype=torch.float)
CLASS_WEIGHTS = CLASS_WEIGHTS * (8 / CLASS_WEIGHTS.sum())

print(f"Train batches : {len(train_loader):,}  ({len(train_ds):,} samples)")
print(f"Val   batches : {len(val_loader):,}  ({len(val_ds):,} samples)")
print(f"Class weights : {CLASS_WEIGHTS.numpy().round(3)}")

Cache: 14,522 records
Train batches : 407  (12,994 samples)
Val   batches : 48  (1,528 samples)
Class weights : [1.029 1.025 1.012 1.04  0.946 1.014 1.032 0.902]


## 3 — Model architecture

In [4]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from peft import LoraConfig, TaskType, get_peft_model

CWE_8_CLASSES = ["CWE-077","CWE-601","CWE-022","CWE-094","CWE-089","CWE-352","CWE-079","unknown"]
CWE_TO_INDEX  = {c: i for i, c in enumerate(CWE_8_CLASSES)}
INDEX_TO_CWE  = {i: c for c, i in CWE_TO_INDEX.items()}


class MultiSampleDropoutHead(nn.Module):
    def __init__(self, hidden, num_classes=8, dropout=0.2, k=5):
        super().__init__()
        mid = hidden // 2
        self.norm     = nn.LayerNorm(hidden)
        self.fc1      = nn.Linear(hidden, mid)
        self.act      = nn.GELU()
        self.dropouts = nn.ModuleList([nn.Dropout(dropout) for _ in range(k)])
        self.fc2      = nn.Linear(mid, num_classes)

    def forward(self, x):
        x = self.act(self.fc1(self.norm(x)))
        if self.training:
            return torch.stack([self.fc2(d(x)) for d in self.dropouts]).mean(0)
        return self.fc2(x)


def focal_label_smooth_loss(logits, targets, class_weights, gamma=2.0, smoothing=0.05):
    n = logits.size(-1)
    logp = F.log_softmax(logits, dim=-1)
    with torch.no_grad():
        true_dist = torch.full_like(logp, smoothing / (n - 1))
        true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)
    ce_per_class = -true_dist * logp
    w    = class_weights.to(logits.device).unsqueeze(0)
    ce_w = (ce_per_class * w).sum(-1)
    pt   = torch.exp(-F.cross_entropy(logits, targets, reduction="none"))
    return (((1 - pt) ** gamma) * ce_w).mean()


class CWEModel(nn.Module):
    """GraphCodeBERT + LoRA + multi-sample dropout head."""

    def __init__(self, model_name, num_cwe=8, lora_r=16, lora_alpha=32,
                 lora_dropout=0.1, class_weights=None, focal_gamma=2.0,
                 label_smoothing=0.05, ms_drop_n=5):
        super().__init__()
        encoder = AutoModel.from_pretrained(model_name)
        encoder.config.use_cache = False
        encoder.gradient_checkpointing_enable()
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
            target_modules=["query", "key", "value"], bias="none",
        )
        self.encoder = get_peft_model(encoder, lora_cfg)
        if hasattr(self.encoder, "enable_input_require_grads"):
            self.encoder.enable_input_require_grads()
        hidden = self.encoder.config.hidden_size
        self.cwe_head = MultiSampleDropoutHead(hidden, num_cwe, dropout=0.2, k=ms_drop_n)
        self.focal_gamma     = focal_gamma
        self.label_smoothing = label_smoothing
        self.register_buffer(
            "class_weights",
            class_weights if class_weights is not None else torch.ones(num_cwe),
        )

    @staticmethod
    def _mean_pool(h, mask):
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

    def encode(self, input_ids, attention_mask):
        """Return mean-pooled embedding (used by EDAT)."""
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self._mean_pool(out.last_hidden_state, attention_mask)

    def classify(self, pooled):
        """Logits from embedding (used by EDAT)."""
        return self.cwe_head(pooled)

    def forward(self, input_ids, attention_mask, cwe_labels=None):
        pooled = self.encode(input_ids, attention_mask)
        logits = self.classify(pooled)
        out = {"logits": logits}
        if cwe_labels is not None:
            out["loss"] = focal_label_smooth_loss(
                logits, cwe_labels, self.class_weights,
                gamma=self.focal_gamma, smoothing=self.label_smoothing,
            )
        return out


def build_model():
    m = CWEModel(
        model_name=cfg["model_name"],
        num_cwe=cfg["num_cwe_classes"],
        lora_r=16, lora_alpha=32, lora_dropout=cfg["lora_dropout"],
        class_weights=CLASS_WEIGHTS,
        focal_gamma=FOCAL_GAMMA,
        label_smoothing=LABEL_SMOOTHING,
        ms_drop_n=MS_DROPOUT_N,
    ).to(DEVICE)
    return m


model = build_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Trainable: 1,184,648 / 125,830,280  (0.94%)


## 4 — EDAT (Embedding-space Data Augmentation Training)

PGD in embedding space:
1. Forward-pass → get `pooled` embedding (no grad through encoder)
2. Add random noise `delta` inside L∞ ball of radius `epsilon`
3. Do `edat_steps` gradient ascent steps on delta to maximise KL divergence
4. Compute KL loss between clean logits and adversarial logits
5. Final loss = focal loss + `kl_weight * KL`

In [5]:
def edat_loss(model, pooled_clean, labels, epsilon, steps, step_size, kl_weight):
    """
    Compute EDAT adversarial KL loss.

    pooled_clean: (B, H) detached embedding from the clean forward pass
    Returns: (focal_loss, adv_kl_loss, total_loss)
    """
    # Clean logits (no grad needed for KL reference)
    with torch.no_grad():
        logits_clean = model.classify(pooled_clean)
        focal = focal_label_smooth_loss(
            logits_clean, labels, model.class_weights,
            gamma=model.focal_gamma, smoothing=model.label_smoothing,
        )
        prob_clean = F.softmax(logits_clean, dim=-1)

    # PGD: find worst-case delta in embedding space
    delta = torch.zeros_like(pooled_clean).uniform_(-epsilon, epsilon)
    delta.requires_grad_(True)

    for _ in range(steps):
        logits_adv = model.classify(pooled_clean + delta)
        # KL(clean || adv) — maximise divergence
        kl = F.kl_div(
            F.log_softmax(logits_adv, dim=-1),
            prob_clean,
            reduction="batchmean",
        )
        kl.backward()
        with torch.no_grad():
            delta.data = (delta + step_size * delta.grad.sign()).clamp(-epsilon, epsilon)
        delta.grad.zero_()

    # Final adversarial KL (with gradient for model params)
    delta_final = delta.detach()
    logits_adv_final = model.classify(pooled_clean + delta_final)
    kl_loss = F.kl_div(
        F.log_softmax(logits_adv_final, dim=-1),
        prob_clean.detach(),
        reduction="batchmean",
    )

    total = focal + kl_weight * kl_loss
    return focal, kl_loss, total


print("EDAT loss function defined.")

EDAT loss function defined.


## 5 — Epsilon Tuning

Grid-search over epsilon values. For each candidate:
- Run a **mini probe training**: 2 epochs on train, evaluate val F1
- Pick epsilon with highest val F1

Small patch size (MAX_LENGTH=128) keeps each probe fast.

In [6]:
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

USE_AMP = DEVICE == "cuda"


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)

    def apply_to(self, model):
        self.backup = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.shadow[n])

    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.backup[n])
        self.backup = {}


def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(ids, mask)
            all_logits.append(out["logits"].float().cpu())
            all_labels.append(batch["cwe_label"])
    logits = torch.cat(all_logits).argmax(-1).numpy()
    labels = torch.cat(all_labels).numpy()
    return f1_score(labels, logits, average="macro", zero_division=0)


def probe_epsilon(epsilon, probe_epochs=2):
    """Train a fresh model for probe_epochs with given epsilon, return val F1."""
    m   = build_model()
    ema = EMA(m, EMA_DECAY)
    opt = torch.optim.AdamW(
        [p for p in m.parameters() if p.requires_grad],
        lr=LR, weight_decay=WEIGHT_DECAY,
    )
    total_steps  = len(train_loader) * probe_epochs
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    for epoch in range(probe_epochs):
        m.train()
        for batch in train_loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["cwe_label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                # Encoder forward, detach embedding for EDAT
                pooled = m.encode(ids, mask)
                _, _, loss = edat_loss(
                    m, pooled.detach(), lbls,
                    epsilon=epsilon,
                    steps=EDAT_STEPS,
                    step_size=EDAT_STEP_SIZE,
                    kl_weight=EDAT_KL_WEIGHT,
                )
                # Also backprop encoder via clean focal loss
                logits_enc = m.classify(pooled)
                enc_loss   = focal_label_smooth_loss(
                    logits_enc, lbls, m.class_weights,
                    gamma=m.focal_gamma, smoothing=m.label_smoothing,
                )
                total_loss = enc_loss + EDAT_KL_WEIGHT * loss

            scaler.scale(total_loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for p in m.parameters() if p.requires_grad], MAX_GRAD_NORM
            )
            scaler.step(opt); scaler.update()
            sched.step()
            ema.update(m)

    ema.apply_to(m)
    f1 = evaluate(m, val_loader)
    ema.restore(m)
    del m, ema, opt, sched, scaler
    torch.cuda.empty_cache()
    return f1


print("Probe function defined.")

Probe function defined.


In [7]:
# ── Run epsilon grid search ──────────────────────────────────────────────────
# Coarse grid first — narrow later if needed
EPSILON_GRID = [0.001, 0.005, 0.01, 0.02, 0.05]
PROBE_EPOCHS = 2   # keep fast — 2 epochs enough to rank epsilons

eps_results = {}
print(f"Tuning epsilon over {EPSILON_GRID} ({PROBE_EPOCHS} probe epochs each)\n")

for eps in EPSILON_GRID:
    t0 = time.time()
    f1 = probe_epsilon(eps, probe_epochs=PROBE_EPOCHS)
    elapsed = time.time() - t0
    eps_results[eps] = f1
    print(f"  epsilon={eps:.4f}  val_F1={f1:.4f}  ({elapsed:.0f}s)")

BEST_EPSILON = max(eps_results, key=eps_results.get)
print(f"\nBest epsilon = {BEST_EPSILON}  (val F1 = {eps_results[BEST_EPSILON]:.4f})")
EDAT_EPSILON = BEST_EPSILON   # override for full training

Tuning epsilon over [0.001, 0.005, 0.01, 0.02, 0.05] (2 probe epochs each)



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epsilon=0.0010  val_F1=0.1927  (238s)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epsilon=0.0200  val_F1=0.1618  (224s)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epsilon=0.0500  val_F1=0.1534  (224s)

Best epsilon = 0.001  (val F1 = 0.1927)


## 6 — Full EDAT Training

In [8]:
from sklearn.metrics import classification_report
import datetime

# Fresh model for full training
model = build_model()
ema   = EMA(model, EMA_DECAY)

trainable_params = [p for p in model.parameters() if p.requires_grad]

try:
    import bitsandbytes as bnb
    optimizer = bnb.optim.PagedAdamW8bit(
        trainable_params, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999)
    )
    print("Optimizer: PagedAdamW8bit")
except Exception as e:
    optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
    print(f"Optimizer: AdamW  ({e})")

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)

BEST_CKPT = CHECKPOINT_DIR / "edat_best.pt"
BEST_F1   = 0.0
no_improve = 0
history    = []

print(f"epsilon={EDAT_EPSILON}  steps={EDAT_STEPS}  step_size={EDAT_STEP_SIZE}  kl_weight={EDAT_KL_WEIGHT}")
print(f"Total training steps: {total_steps:,}  warmup: {warmup_steps}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Optimizer: PagedAdamW8bit
epsilon=0.001  steps=3  step_size=0.005  kl_weight=1.0
Total training steps: 3,256  warmup: 325


In [9]:
def collect_logits(model, loader):
    model.eval()
    all_logits, all_labels, total_loss, n = [], [], 0.0, 0
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["cwe_label"].to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                out = model(ids, mask, cwe_labels=lbls)
            total_loss += out["loss"].item(); n += 1
            all_logits.append(out["logits"].float().cpu())
            all_labels.append(lbls.cpu())
    return torch.cat(all_logits), torch.cat(all_labels), total_loss / max(n, 1)


for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = epoch_focal = epoch_kl = 0.0
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True)

    for step, batch in enumerate(pbar, 1):
        ids  = batch["input_ids"].to(DEVICE, non_blocking=True)
        mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        lbls = batch["cwe_label"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            # ── Step 1: encoder forward ──────────────────────────────────────
            pooled = model.encode(ids, mask)          # (B, H)

            # ── Step 2: EDAT adversarial KL on head only (detached pooled) ──
            focal_h, kl_h, _ = edat_loss(
                model, pooled.detach(), lbls,
                epsilon=EDAT_EPSILON,
                steps=EDAT_STEPS,
                step_size=EDAT_STEP_SIZE,
                kl_weight=EDAT_KL_WEIGHT,
            )

            # ── Step 3: encoder-aware clean focal loss ───────────────────────
            logits_clean = model.classify(pooled)     # grad flows to encoder
            focal_enc = focal_label_smooth_loss(
                logits_clean, lbls, model.class_weights,
                gamma=model.focal_gamma, smoothing=model.label_smoothing,
            )

            # ── Total loss ───────────────────────────────────────────────────
            loss = focal_enc + EDAT_KL_WEIGHT * kl_h

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
        scaler.step(optimizer); scaler.update()
        scheduler.step()
        ema.update(model)

        epoch_loss  += loss.item()
        epoch_focal += focal_enc.item()
        epoch_kl    += kl_h.item()

        if step % 50 == 0:
            pbar.set_postfix({
                "loss":  f"{loss.item():.4f}",
                "focal": f"{focal_enc.item():.4f}",
                "kl":    f"{kl_h.item():.4f}",
                "lr":    f"{scheduler.get_last_lr()[0]:.2e}",
            })

    # ── Validate with EMA weights ────────────────────────────────────────────
    ema.apply_to(model)
    val_logits, val_labels, val_loss = collect_logits(model, val_loader)
    preds  = val_logits.argmax(-1).numpy()
    y      = val_labels.numpy()
    val_f1 = f1_score(y, preds, average="macro", zero_division=0)
    ema.restore(model)

    n_steps = len(train_loader)
    elapsed = time.time() - t0
    print(f"\nEpoch {epoch:02d}  "
          f"train_loss={epoch_loss/n_steps:.4f}  "
          f"(focal={epoch_focal/n_steps:.4f} kl={epoch_kl/n_steps:.4f})  "
          f"val_loss={val_loss:.4f}  val_F1={val_f1:.4f}  ({elapsed:.0f}s)")
    print(classification_report(y, preds, target_names=CWE_8_CLASSES, zero_division=0))

    history.append({
        "epoch": epoch,
        "train_loss":  round(epoch_loss / n_steps, 4),
        "focal_loss":  round(epoch_focal / n_steps, 4),
        "kl_loss":     round(epoch_kl / n_steps, 4),
        "val_loss":    round(val_loss, 4),
        "val_f1":      round(val_f1, 4),
        "elapsed_s":   round(elapsed, 1),
    })

    if val_f1 > BEST_F1:
        BEST_F1    = val_f1
        no_improve = 0
        torch.save({
            "epoch": epoch, "val_f1": val_f1,
            "model_state_dict": model.state_dict(),
            "ema_shadow": {k: v.cpu() for k, v in ema.shadow.items()},
            "config": cfg,
            "edat": {
                "epsilon":   EDAT_EPSILON,
                "steps":     EDAT_STEPS,
                "step_size": EDAT_STEP_SIZE,
                "kl_weight": EDAT_KL_WEIGHT,
            },
        }, BEST_CKPT)
        print(f"  ✓ New best F1={val_f1:.4f} — saved")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print(f"Early stop @ epoch {epoch}"); break

print(f"\nBest val F1 (EMA, argmax) = {BEST_F1:.4f}")

Epoch 1/8:   0%|          | 0/407 [00:00<?, ?it/s]


Epoch 01  train_loss=1.3900  (focal=1.3875 kl=0.0024)  val_loss=1.5487  val_F1=0.0273  (114s)
              precision    recall  f1-score   support

     CWE-077       0.00      0.00      0.00       153
     CWE-601       0.00      0.00      0.00        93
     CWE-022       0.00      0.00      0.00       103
     CWE-094       0.08      0.98      0.15       124
     CWE-089       0.00      0.00      0.00       350
     CWE-352       0.00      0.00      0.00       146
     CWE-079       0.33      0.01      0.01       152
     unknown       0.35      0.03      0.05       407

    accuracy                           0.09      1528
   macro avg       0.10      0.13      0.03      1528
weighted avg       0.13      0.09      0.03      1528

  ✓ New best F1=0.0273 — saved


Epoch 2/8:   0%|          | 0/407 [00:00<?, ?it/s]


Epoch 02  train_loss=0.8550  (focal=0.8512 kl=0.0038)  val_loss=1.4666  val_F1=0.1247  (126s)
              precision    recall  f1-score   support

     CWE-077       0.50      0.01      0.01       153
     CWE-601       0.21      0.17      0.19        93
     CWE-022       0.33      0.02      0.04       103
     CWE-094       0.10      0.69      0.17       124
     CWE-089       0.78      0.05      0.10       350
     CWE-352       0.29      0.01      0.03       146
     CWE-079       0.27      0.02      0.04       152
     unknown       0.38      0.50      0.43       407

    accuracy                           0.22      1528
   macro avg       0.36      0.18      0.12      1528
weighted avg       0.43      0.22      0.17      1528

  ✓ New best F1=0.1247 — saved


Epoch 3/8:   0%|          | 0/407 [00:00<?, ?it/s]

...


Epoch 06  train_loss=0.3958  (focal=0.3916 kl=0.0042)  val_loss=0.8292  val_F1=0.5213  (126s)
              precision    recall  f1-score   support

     CWE-077       0.48      0.39      0.43       153
     CWE-601       0.36      0.51      0.42        93
     CWE-022       0.33      0.50      0.40       103
     CWE-094       0.36      0.50      0.42       124
     CWE-089       0.81      0.73      0.77       350
     CWE-352       0.59      0.69      0.64       146
     CWE-079       0.45      0.32      0.38       152
     unknown       0.78      0.68      0.73       407

    accuracy                           0.59      1528
   macro avg       0.52      0.54      0.52      1528
weighted avg       0.62      0.59      0.60      1528

  ✓ New best F1=0.5213 — saved


Epoch 7/8:   0%|          | 0/407 [00:00<?, ?it/s]


Epoch 07  train_loss=0.3691  (focal=0.3649 kl=0.0042)  val_loss=0.8132  val_F1=0.5363  (125s)
              precision    recall  f1-score   support

     CWE-077       0.47      0.42      0.44       153
     CWE-601       0.39      0.53      0.45        93
     CWE-022       0.36      0.48      0.41       103
     CWE-094       0.38      0.48      0.42       124
     CWE-089       0.81      0.73      0.77       350
     CWE-352       0.59      0.74      0.66       146
     CWE-079       0.45      0.36      0.40       152
     unknown       0.80      0.69      0.74       407

    accuracy                           0.60      1528
   macro avg       0.53      0.55      0.54      1528
weighted avg       0.63      0.60      0.61      1528

  ✓ New best F1=0.5363 — saved


Epoch 8/8:   0%|          | 0/407 [00:00<?, ?it/s]


Epoch 08  train_loss=0.3567  (focal=0.3524 kl=0.0042)  val_loss=0.8102  val_F1=0.5423  (126s)
              precision    recall  f1-score   support

     CWE-077       0.46      0.40      0.43       153
     CWE-601       0.41      0.56      0.47        93
     CWE-022       0.39      0.52      0.45       103
     CWE-094       0.38      0.47      0.42       124
     CWE-089       0.80      0.73      0.77       350
     CWE-352       0.59      0.74      0.65       146
     CWE-079       0.44      0.37      0.40       152
     unknown       0.81      0.69      0.75       407

    accuracy                           0.61      1528
   macro avg       0.54      0.56      0.54      1528
weighted avg       0.63      0.61      0.61      1528

  ✓ New best F1=0.5423 — saved

Best val F1 (EMA, argmax) = 0.5423


## 7 — Per-class threshold tuning

In [ ]:
from sklearn.metrics import precision_score, recall_score

# Reload best checkpoint with EMA weights applied
ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
for n, p in model.named_parameters():
    if p.requires_grad and n in ckpt["ema_shadow"]:
        p.data.copy_(ckpt["ema_shadow"][n].to(DEVICE))

val_logits, val_labels, _ = collect_logits(model, val_loader)
probs = torch.softmax(val_logits, dim=-1).numpy()
y     = val_labels.numpy()
C     = probs.shape[1]


def predict_with_thresholds(probs, thresholds):
    preds = np.full(len(probs), -1, dtype=np.int64)
    fires = probs >= thresholds[None, :]
    for i in range(len(probs)):
        idx = np.where(fires[i])[0]
        preds[i] = idx[probs[i, idx].argmax()] if len(idx) else probs[i].argmax()
    return preds


best_thresh = np.full(C, 0.5)
for c in range(C):
    best_f1_c = -1
    for t in np.arange(0.05, 0.96, 0.02):
        cand = best_thresh.copy(); cand[c] = t
        f1c  = f1_score(y, predict_with_thresholds(probs, cand),
                        labels=[c], average="macro", zero_division=0)
        if f1c > best_f1_c:
            best_f1_c = f1c; best_thresh[c] = t

tuned_preds = predict_with_thresholds(probs, best_thresh)
tuned_f1    = f1_score(y, tuned_preds, average="macro", zero_division=0)
tuned_p     = precision_score(y, tuned_preds, average="macro", zero_division=0)
tuned_r     = recall_score(y, tuned_preds, average="macro", zero_division=0)

print("Per-class thresholds:")
for c, t in zip(CWE_8_CLASSES, best_thresh): print(f"  {c:<10} {t:.2f}")
print(f"\nArgmax F1 : {BEST_F1:.4f}")
print(f"Tuned  F1 : {tuned_f1:.4f}  P={tuned_p:.4f}  R={tuned_r:.4f}")
print(classification_report(y, tuned_preds, target_names=CWE_8_CLASSES, zero_division=0))

ckpt["thresholds"]    = best_thresh.tolist()
ckpt["val_f1_argmax"] = float(BEST_F1)
ckpt["val_f1_tuned"]  = float(tuned_f1)
torch.save(ckpt, BEST_CKPT)
print(f"Saved thresholds → {BEST_CKPT}")

Per-class thresholds:
  CWE-077    0.27
  CWE-601    0.33
  CWE-022    0.31
  CWE-094    0.59
  CWE-089    0.31
  CWE-352    0.35
  CWE-079    0.17
  unknown    0.23

Argmax F1 : 0.5423
Tuned  F1 : 0.5561  P=0.5488  R=0.5709
              precision    recall  f1-score   support

     CWE-077       0.49      0.46      0.48       153
     CWE-601       0.42      0.55      0.48        93
     CWE-022       0.39      0.52      0.45       103
     CWE-094       0.41      0.44      0.42       124
     CWE-089       0.81      0.73      0.77       350
     CWE-352       0.61      0.73      0.67       146
     CWE-079       0.45      0.42      0.44       152
     unknown       0.80      0.71      0.76       407

    accuracy                           0.62      1528
   macro avg       0.55      0.57      0.56      1528
weighted avg       0.64      0.62      0.63      1528



## 8 — Final report + log

In [12]:
log = {
    "run_date":      datetime.datetime.now().isoformat(),
    "device":        DEVICE,
    "model_name":    cfg["model_name"],
    "max_length":    MAX_LENGTH,
    "batch_size":    BATCH_SIZE,
    "epochs":        EPOCHS,
    "edat_epsilon":  EDAT_EPSILON,
    "edat_steps":    EDAT_STEPS,
    "edat_step_size":EDAT_STEP_SIZE,
    "edat_kl_weight":EDAT_KL_WEIGHT,
    "epsilon_sweep": {str(k): round(v, 4) for k, v in eps_results.items()},
    "val_f1_argmax": float(BEST_F1),
    "val_f1_tuned":  float(tuned_f1),
    "thresholds":    best_thresh.tolist(),
    "history":       history,
}
ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = LOG_DIR / f"edat_run_{ts}.json"
with open(log_path, "w") as f:
    json.dump(log, f, indent=2)
print(f"Log saved: {log_path}")

print(f"\n{'Epoch':>5} {'TrainLoss':>10} {'FocalLoss':>10} {'KLLoss':>8} {'ValF1':>8}")
for r in history:
    print(f"{r['epoch']:>5} {r['train_loss']:>10.4f} {r['focal_loss']:>10.4f} "
          f"{r['kl_loss']:>8.4f} {r['val_f1']:>8.4f}")

Log saved: /content/drive/MyDrive/CSI_Project/logs/edat_run_20260427_164144.json

Epoch  TrainLoss  FocalLoss   KLLoss    ValF1
    1     1.3900     1.3875   0.0024   0.0273
    2     0.8550     0.8512   0.0038   0.1247
    3     0.6428     0.6386   0.0042   0.3118
    4     0.5289     0.5247   0.0042   0.4407
    5     0.4577     0.4534   0.0043   0.4824
    6     0.3958     0.3916   0.0042   0.5213
    7     0.3691     0.3649   0.0042   0.5363
    8     0.3567     0.3524   0.0042   0.5423
